# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: 
* Model: 
* Evaluation approach: 
* Fine-tuning dataset: 

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset. 

PEFT technique: LoRA (Low-Rank Adaptation)
Model: GPT-2
Evaluation approach: Hugging Face Trainer with accuracy metric
Fine-tuning dataset: mteb/amazon_reviews_multi (English, binary sentiment classification)

In [2]:
import torch
import numpy as np
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer, TrainingArguments, Trainer
from datasets import load_dataset

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

model = GPT2ForSequenceClassification.from_pretrained('gpt2', num_labels=2)
model.config.pad_token_id = tokenizer.eos_token_id

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {'accuracy': accuracy}

print("Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!


In [3]:
dataset = load_dataset('mteb/amazon_reviews_multi', 'en')

def filter_and_label(example):
    if example['label'] == 2:
        return False
    return True

def simplify_labels(example):
    example['labels'] = 0 if example['label'] < 2 else 1
    return example

dataset = dataset.filter(filter_and_label)
dataset = dataset.map(simplify_labels)

small_train = dataset['train'].shuffle(seed=42).select(range(500))
small_eval = dataset['validation'].shuffle(seed=42).select(range(100))

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

small_train = small_train.map(tokenize, batched=True)
small_eval = small_eval.map(tokenize, batched=True)

small_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
small_eval.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print("Dataset ready!")
print(f"Training samples: {len(small_train)}")
print(f"Evaluation samples: {len(small_eval)}")

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset ready!
Training samples: 500
Evaluation samples: 100


In [6]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_eval_batch_size=8,
    no_cuda=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

print("Evaluating original GPT-2 on Amazon Reviews...")
original_results = trainer.evaluate()
print(f"Original Model Accuracy: {original_results['eval_accuracy']:.4f}")

Evaluating original GPT-2 on Amazon Reviews...


Original Model Accuracy: 0.4400


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [6]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['c_attn']
)

peft_model = get_peft_model(model, lora_config)
peft_model.config.pad_token_id = tokenizer.eos_token_id
peft_model.print_trainable_parameters()

/opt/conda/lib/python3.10/site-packages/peft/tuners/lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 592,896 || all params: 125,032,704 || trainable%: 0.47419273600609324


In [8]:
peft_training_args = TrainingArguments(
    output_dir='./peft_results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-4,
    no_cuda=False,
    load_best_model_at_end=True,
    save_steps=500,
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning on Amazon Reviews...")
peft_trainer.train()
print("Fine-tuning complete!")

Starting fine-tuning on Amazon Reviews...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.602442,0.650000
2,No log,0.544727,0.750000
3,No log,0.519569,0.780000


Fine-tuning complete!


In [9]:
peft_model.save_pretrained('/tmp/saved_peft_model')
print("Model saved!")

import os
print(os.listdir('/tmp/saved_peft_model'))

Model saved!
['adapter_model.bin', 'adapter_config.json', 'README.md']


###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [ ]:
# Saving the model
model.save("/tmp/your_model_name")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [10]:
from peft import AutoPeftModelForsequenceClassification 

loaded_peft_model = AutoPeftModelForSequenceClassification.from_pretrained('./peft_results/checkpoint-189')
loaded_peft_model.config.pad_token_id = tokenizer.eos_token_id

print('Fine-tuned model loaded successfully!')

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fine-tuned model loaded successfully!


In [11]:
fine_tuned_trainer = Trainer(
    model=loaded_peft_model,
    args=training_args,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

print("Evaluating fine-tuned model on Amazon Reviews...")
finetuned_results = fine_tuned_trainer.evaluate()
print(f"Fine-Tuned Model Accuracy: {finetuned_results['eval_accuracy']:.4f}")

print("\n========== RESULTS COMPARISON ==========")
print(f"Dataset:                     Amazon Reviews Multi (English)")
print(f"Task:                        Sentiment Classification (Positive vs Negative)")
print(f"PEFT Technique:              LoRA (r=16, alpha=16)")
print(f"Original GPT-2 Accuracy:     {original_results['eval_accuracy']:.4f}")
print(f"Fine-Tuned GPT-2 Accuracy:   {finetuned_results['eval_accuracy']:.4f}")
improvement = finetuned_results['eval_accuracy'] - original_results['eval_accuracy']
print(f"Improvement:                 +{improvement:.4f}")
print("=========================================")

Evaluating fine-tuned model on Amazon Reviews...


Fine-Tuned Model Accuracy: 0.7800

========== RESULTS COMPARISON ==========
Dataset:                     Amazon Reviews Multi (English)
Task:                        Sentiment Classification (Positive vs Negative)
PEFT Technique:              LoRA (r=16, alpha=16)
Original GPT-2 Accuracy:     0.4400
Fine-Tuned GPT-2 Accuracy:   0.7800
Improvement:                 +0.3400
